# 🚀 Zalo AI Challenge 2026: DataMate — SFT Dataset Expansion with Qwen 7B + LoRA
## Sinh và Sàng lọc Dataset SFT chuẩn (+1đ) bằng Qwen2.5-Coder-7B-Instruct + LoRA Adapter

> **Điểm nâng cấp quan trọng:**
> - Tự động nạp **LoRA Adapter đã huấn luyện** tại `result/sft_adapter_qwen7b/final_adapter` (hoặc trên Google Drive).
> - Mô hình sau SFT có khả năng sinh code giải nghiệp vụ doanh nghiệp chuẩn xác hơn nhiều so với base model.
> - Thực thi qua Sandbox và sàng lọc 4 Cổng khắt khe: AST đọc file thật, exit code 0, đối chiếu 100% khớp đáp án BTC (+1.0 điểm).
> - Tự động sao lưu dataset mở rộng về Google Drive sau mỗi batch.


Đầu vào v1: BUSINESS_DIR + README + file/sheet + toàn bộ cột/dtype + câu hỏi + as_of_date. Không đưa dòng dữ liệu mẫu, unique values, full rulebook hay checklist lặp vào prompt. Code đọc quy tắc từ RULEBOOK_PATH khi cần. Ghi kết quả vào thư mục mới để giữ bộ dữ liệu cũ.

### 1. Cài đặt thư viện phụ thuộc (Dependencies)

In [ ]:
# Cài đặt các thư viện cần thiết cho DataMate, vLLM, Transformers và WandB
!pip install -q "pandas>=2.2.0" numpy openpyxl python-calamine xlrd duckdb pyyaml tqdm pydantic
!pip install -q torch transformers accelerate peft bitsandbytes wandb vllm
!pip uninstall -y torchao -q


### 2. Thiết lập Môi trường & Kết nối Google Drive / Repo

In [ ]:
import os
import sys
from pathlib import Path

# Tự động nhận diện Google Colab và Mount Drive để lưu trữ dữ liệu an toàn
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    
    # Tìm kiếm thư mục project trên Drive hoặc /content
    candidates = [
        Path('/content/datamate'),
        Path('/content/ZALO-AI-CHALLENGE-2026'),
        Path('/content/drive/MyDrive/ZALO-AI-CHALLENGE-2026'),
        Path('/content/drive/MyDrive/Colab Notebooks/ZALO-AI-CHALLENGE-2026'),
        Path('/content'),
    ]
    PROJECT_ROOT = Path('/content')
    for cand in candidates:
        if cand.exists() and (cand / "data").exists():
            PROJECT_ROOT = cand
            break
else:
    PROJECT_ROOT = Path('.').resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Chuyển thư mục làm việc về PROJECT_ROOT
os.chdir(str(PROJECT_ROOT))

print(f"Thư mục làm việc: {PROJECT_ROOT}")
print(f"Đã tìm thấy thư mục data/train: {(PROJECT_ROOT / 'data' / 'train').exists()}")
print(f"CUDA có sẵn: {os.system('nvidia-smi') == 0}")


### 3. Đăng nhập Weights & Biases (Tùy chọn theo dõi trực quan)

In [ ]:
import wandb

wandb_api_key = os.environ.get("WANDB_API_KEY")
if not wandb_api_key and IN_COLAB:
    try:
        from google.colab import userdata
        wandb_api_key = userdata.get('WANDB_API_KEY')
    except Exception:
        pass

if wandb_api_key:
    os.environ["WANDB_API_KEY"] = wandb_api_key
    wandb.login(key=wandb_api_key)
    print("✅ Logged in to Weights & Biases via API Key.")
else:
    print("ℹ️ Chưa cấu hình WANDB_API_KEY. Pipeline vẫn chạy và xuất log JSONL nội bộ bình thường.")


### 4. Kiểm tra & Tự động Làm sạch VRAM GPU (Khuyến nghị)
> 💡 **Mẹo:** Nếu trước đó bạn đã từng nạp model qua PyTorch/HuggingFace mà chưa khởi động lại session, GPU có thể vẫn bị chiếm dụng VRAM. Cell dưới đây sẽ tự động kiểm tra và giải phóng sạch GPU trước khi khởi chạy vLLM.

In [ ]:
import torch

if torch.cuda.is_available():
    total_vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    allocated = torch.cuda.memory_allocated() / (1024**3)
    reserved = torch.cuda.memory_reserved() / (1024**3)
    gpu_name = torch.cuda.get_device_name(0)
    print(f"GPU: {gpu_name} (Tổng VRAM: {total_vram:.1f} GB)")
    print(f"VRAM đang dùng (Allocated): {allocated:.2f} GB | Reserved: {reserved:.2f} GB")
    
    if allocated > 2.0:
        print("⚠️ VRAM đang bị chiếm dụng bởi model cũ! Đang tự động làm sạch kernel...")
        import os
        os.kill(os.getpid(), 9)
    else:
        print("✅ GPU hoàn toàn sạch sẽ, sẵn sàng nạp vLLM!")
else:
    print("❌ Không tìm thấy GPU CUDA! Vui lòng chọn Runtime -> Change runtime type -> A100 GPU.")


### 5. Sinh dataset theo batch bằng vLLM kết hợp LoRA Adapter

- Tự động tìm kiếm adapter LoRA tại `result/sft_adapter_qwen7b/final_adapter` trên Google Drive hoặc local SSD.
- vLLM nạp model base và kích hoạt LoRA request (`enable_lora=True`) để sinh code giải theo batch.
- Sandbox chạy tối đa 2 worker CPU, timeout 35 giây/câu để chấm điểm.
- Chỉ những câu đạt điểm tuyệt đối (+1.0đ) và đọc file thật mới được lưu vào `verified_sft_dataset.jsonl`.


In [ ]:
import os, subprocess, sys, shutil
from pathlib import Path

# 1. Đồng bộ mã nguồn mới nhất từ Google Drive
source_root = Path('/content/drive/MyDrive/Colab Notebooks/ZALO-AI-CHALLENGE-2026')
for relative in [
    'scripts/generate_sft_vllm.py',
    'scripts/mine_and_expand_sft.py',
    'src/agent/prompts.py',
    'src/agent/solver.py',
    'src/profiler/business_loader.py',
    'src/time_resolver/calendar_utils.py',
]:
    source, target = source_root / relative, PROJECT_ROOT / relative
    if source.is_file() and source.resolve() != target.resolve():
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, target)

# 2. Tự động tìm kiếm & sao chép LoRA Adapter đã huấn luyện sang SSD để tối ưu I/O
adapter_candidates = [
    PROJECT_ROOT / 'result/sft_adapter_qwen7b/final_adapter',
    source_root / 'result/sft_adapter_qwen7b/final_adapter',
    PROJECT_ROOT / 'result/sft_adapter_qwen7b',
    source_root / 'result/sft_adapter_qwen7b',
]
chosen_adapter = None
for cand in adapter_candidates:
    if (cand / 'adapter_config.json').exists():
        chosen_adapter = cand
        break

adapter_args = []
if chosen_adapter:
    # Sao chép sang SSD cục bộ (/content/datamate) để vLLM nạp nhanh nhất
    local_adapter = PROJECT_ROOT / 'result/sft_adapter_qwen7b/final_adapter'
    if chosen_adapter.resolve() != local_adapter.resolve():
        local_adapter.mkdir(parents=True, exist_ok=True)
        for item in chosen_adapter.iterdir():
            if item.is_file():
                shutil.copy2(item, local_adapter / item.name)
        chosen_adapter = local_adapter
    print(f'🎯 Đã nạp thành công LoRA Adapter: {chosen_adapter}')
    adapter_args = ['--lora-adapter', str(chosen_adapter)]
else:
    print('⚠️ Không tìm thấy adapter LoRA đã train; sẽ chạy bằng Qwen 7B base model.')

# 3. Kiểm tra Model Base trên máy cục bộ (nếu có tải sẵn)
base_path = Path('/content/models/Qwen2.5-Coder-7B-Instruct')
base_model = str(base_path) if base_path.exists() else 'Qwen/Qwen2.5-Coder-7B-Instruct'
print(f'🤖 Model Base: {base_model}')

# 4. Khởi chạy tiến trình sinh dataset vLLM
command = [
    sys.executable, '-u', 'scripts/generate_sft_vllm.py',
    '--model-id', base_model,
    '--output-dir', 'result/sft_dataset_qwen7b_folder_schema_v1',
    '--drive-backup-dir', '/content/drive/MyDrive/datamate_sft_backup_qwen7b_folder_schema_v1',
    '--batch-size', '32',
    '--gpu-util', '0.88',
    '--max-model-len', '16384',
    '--max-tokens', '2048',
    '--sandbox-workers', '2',
    '--sandbox-timeout', '35',
] + adapter_args

with subprocess.Popen(command, cwd=PROJECT_ROOT, stdout=subprocess.PIPE,
                      stderr=subprocess.STDOUT, text=True, encoding='utf-8', errors='replace') as process:
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
        return_code = process.wait()
    except KeyboardInterrupt:
        process.terminate()
        process.wait()
        raise
if return_code:
    raise RuntimeError(f'Stage 1 failed (exit {return_code}); see output above.')


### 6. Kiểm tra & Thống kê Dataset SFT Thu Được

In [ ]:
import json
from pathlib import Path

sft_file = PROJECT_ROOT / "result" / "sft_dataset_qwen7b_folder_schema_v1" / "verified_sft_dataset.jsonl"
if not sft_file.exists():
    drive_file = Path("/content/drive/MyDrive/datamate_sft_backup_qwen7b_folder_schema_v1/verified_sft_dataset.jsonl")
    if drive_file.exists():
        sft_file = drive_file

if sft_file.exists():
    samples = []
    with open(sft_file, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                samples.append(json.loads(line))
                
    print(f"🎉 Tổng số mẫu SFT đạt chuẩn (+1.0 điểm): {len(samples)} mẫu")
    print(f"📂 File lưu trữ: {sft_file} ({sft_file.stat().st_size / (1024**2):.2f} MB)\n")
    
    # In mẫu đầu tiên làm ví dụ kiểm tra Chat Template
    sample = samples[0]
    print(f"--- MẪU VÍ DỤ ({sample.get('id')} - {sample.get('business_id')}) ---")
    for msg in sample.get("messages", []):
        role = msg.get("role")
        content = msg.get("content", "")
        print(f"\n[{role.upper()}] ({len(content)} ký tự):")
        if role == "assistant":
            print(content)
        else:
            print(content[:300] + "...")
else:
    print(f"Chưa tìm thấy file {sft_file}. Hãy chạy cell 5 ở trên!")


### 7. Tự động ngắt kết nối session (tùy chọn)
Đặt `AUTO_SHUTDOWN = True` nếu muốn giải phóng session Colab sau khi sinh xong các câu đã chọn.


In [ ]:
AUTO_SHUTDOWN = False  # Đổi thành True nếu muốn tự động tắt máy sau khi xong

if AUTO_SHUTDOWN and IN_COLAB:
    print("🔌 Đang tự động ngắt kết nối session Colab để tiết kiệm Compute Units...")
    from google.colab import runtime
    runtime.unassign()
else:
    print("Session vẫn được duy trì.")
